# Session 4.3: Feature Selection
## Module 4: Model Evaluation & Optimization

**Learning Objectives:**
- Understand why feature selection matters for model performance
- Use feature importance from tree-based models
- Apply Recursive Feature Elimination (RFE)
- Use correlation-based feature selection
- Leverage L1 regularization for automatic feature selection
- Visualize performance vs number of features
- Build models with optimal feature subsets

**Why Feature Selection Matters:**
More features ≠ better models! Irrelevant features add noise, increase training time, and can cause overfitting. Good feature selection:
- Improves model performance
- Reduces overfitting
- Speeds up training
- Makes models more interpretable

**What You'll Build:**
- Compare 4 feature selection methods
- Find optimal number of features
- Visualize feature importance
- Compare model performance before/after feature selection

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.feature_selection import RFE, SelectKBest, f_classif, mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use('seaborn-v0_8-darkgrid')

print('✅ Libraries imported!')
print(f'Random seed: {RANDOM_STATE}')

## Step 2: Create Dataset with Relevant and Irrelevant Features

In [ ]:
print('CREATING DATASET WITH MIXED FEATURE QUALITY')
print('='*60)

# Create dataset with clear signal
X, y = make_classification(
    n_samples=1000,
    n_features=50,
    n_informative=15,
    n_redundant=10,
    n_repeated=5,
    n_classes=2,
    random_state=RANDOM_STATE
)

print(f'Total features: {X.shape[1]}')
print(f'  Informative: 15')
print(f'  Redundant: 10')
print(f'  Repeated: 5')
print(f'  Noise: 20')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'\nTraining samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')
print('\n💡 Goal: Select the 15-20 most important features')

## Step 3: Baseline - All Features

> **💡 AI Assistant Prompt:**
> 
> "Explain why using all features can hurt model performance. Show code demonstrating the curse of dimensionality with irrelevant features."

In [ ]:
print('BASELINE: Using All 50 Features')
print('='*60)

rf_baseline = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_baseline.fit(X_train, y_train)

baseline_train_score = rf_baseline.score(X_train, y_train)
baseline_test_score = rf_baseline.score(X_test, y_test)
baseline_f1 = f1_score(y_test, rf_baseline.predict(X_test))

print(f'Training accuracy: {baseline_train_score:.4f}')
print(f'Test accuracy: {baseline_test_score:.4f}')
print(f'Test F1-score: {baseline_f1:.4f}')
print(f'Train/Test gap: {baseline_train_score - baseline_test_score:.4f}')

if baseline_train_score - baseline_test_score > 0.05:
    print('\n⚠️ Significant overfitting detected!')
    print('   Feature selection should help reduce this gap.')

baseline_scores = {
    'train': baseline_train_score,
    'test': baseline_test_score,
    'f1': baseline_f1
}

## Step 4: Method 1 - Feature Importance (Tree-Based)

Tree-based models like Random Forest calculate feature importance based on how much each feature contributes to reducing impurity.

In [ ]:
print('METHOD 1: Feature Importance from Random Forest')
print('='*60)

importances = rf_baseline.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': [f'Feature_{i}' for i in range(len(importances))],
    'Importance': importances
}).sort_values('Importance', ascending=False)

print('\nTop 15 Most Important Features:')
print(feature_importance_df.head(15).to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 20 features
top_20 = feature_importance_df.head(20)
axes[0].barh(range(len(top_20)), top_20['Importance'], color='#3498db')
axes[0].set_yticks(range(len(top_20)))
axes[0].set_yticklabels(top_20['Feature'])
axes[0].set_xlabel('Importance', fontsize=11)
axes[0].set_title('Top 20 Feature Importances', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()

# Cumulative importance
cumulative_importance = np.cumsum(feature_importance_df['Importance'].values)
axes[1].plot(range(1, len(cumulative_importance)+1), cumulative_importance, linewidth=2)
axes[1].axhline(y=0.9, color='r', linestyle='--', label='90% threshold')
axes[1].axhline(y=0.95, color='orange', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Number of Features', fontsize=11)
axes[1].set_ylabel('Cumulative Importance', fontsize=11)
axes[1].set_title('Cumulative Feature Importance', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

n_features_90 = np.argmax(cumulative_importance >= 0.9) + 1
n_features_95 = np.argmax(cumulative_importance >= 0.95) + 1

print(f'\nFeatures for 90% importance: {n_features_90}')
print(f'Features for 95% importance: {n_features_95}')
print(f'Reduction: {50 - n_features_95} features can be dropped!')

## Step 5: Method 2 - Recursive Feature Elimination (RFE)

RFE recursively removes features and builds models until the specified number of features is reached.

In [ ]:
print('METHOD 2: Recursive Feature Elimination (RFE)')
print('='*60)

rf_rfe = RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE)
rfe = RFE(estimator=rf_rfe, n_features_to_select=15, step=5)

print('\nRunning RFE to select 15 features...')
rfe.fit(X_train, y_train)

selected_features_rfe = [f'Feature_{i}' for i in range(50) if rfe.support_[i]]
print(f'\nSelected features: {len(selected_features_rfe)}')
print(f'Feature rankings (1=selected): {np.unique(rfe.ranking_, return_counts=True)}')

# Test performance with selected features
X_train_rfe = rfe.transform(X_train)
X_test_rfe = rfe.transform(X_test)

rf_rfe_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_rfe_model.fit(X_train_rfe, y_train)

rfe_test_score = rf_rfe_model.score(X_test_rfe, y_test)
rfe_f1 = f1_score(y_test, rf_rfe_model.predict(X_test_rfe))

print(f'\nTest accuracy with 15 features: {rfe_test_score:.4f}')
print(f'Test F1-score: {rfe_f1:.4f}')
print(f'Baseline (50 features): {baseline_test_score:.4f}')
print(f'Improvement: {(rfe_test_score - baseline_test_score):.4f}')

## Step 6: Method 3 - Correlation-Based Selection

Select features based on their correlation with the target variable.

In [ ]:
print('METHOD 3: Correlation-Based Feature Selection')
print('='*60)

# Use SelectKBest with f_classif (ANOVA F-value)
selector = SelectKBest(score_func=f_classif, k=15)
selector.fit(X_train, y_train)

# Get scores
scores = selector.scores_
feature_scores_df = pd.DataFrame({
    'Feature': [f'Feature_{i}' for i in range(50)],
    'Score': scores
}).sort_values('Score', ascending=False)

print('\nTop 15 Features by F-score:')
print(feature_scores_df.head(15).to_string(index=False))

# Transform and test
X_train_corr = selector.transform(X_train)
X_test_corr = selector.transform(X_test)

rf_corr = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_corr.fit(X_train_corr, y_train)

corr_test_score = rf_corr.score(X_test_corr, y_test)
corr_f1 = f1_score(y_test, rf_corr.predict(X_test_corr))

print(f'\nTest accuracy with 15 features: {corr_test_score:.4f}')
print(f'Test F1-score: {corr_f1:.4f}')
print(f'Baseline (50 features): {baseline_test_score:.4f}')
print(f'Improvement: {(corr_test_score - baseline_test_score):.4f}')

# Visualize scores
plt.figure(figsize=(12, 5))
plt.bar(range(50), scores, color='#2ecc71', alpha=0.7)
plt.axhline(y=np.sort(scores)[-15], color='r', linestyle='--', label='Selection threshold')
plt.xlabel('Feature Index', fontsize=11)
plt.ylabel('F-score', fontsize=11)
plt.title('Feature Scores (F-classif)', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 7: Method 4 - L1 Regularization (Lasso)

L1 regularization automatically sets some feature weights to zero, effectively performing feature selection.

> **💡 AI Assistant Prompt:**
> 
> "Explain how L1 regularization performs automatic feature selection. Show code using Lasso for feature selection and visualize which features have non-zero coefficients."

In [ ]:
print('METHOD 4: L1 Regularization (Lasso)')
print('='*60)

# Scale features (important for Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Lasso with CV to find optimal alpha
lasso = LogisticRegression(penalty='l1', solver='saga', random_state=RANDOM_STATE, max_iter=1000)

# Try different C values (inverse of regularization strength)
C_values = [0.001, 0.01, 0.1, 1, 10]
best_score = 0
best_C = None

for C in C_values:
    lasso.C = C
    lasso.fit(X_train_scaled, y_train)
    score = lasso.score(X_test_scaled, y_test)
    if score > best_score:
        best_score = score
        best_C = C

print(f'Best C value: {best_C}')
print(f'Best accuracy: {best_score:.4f}')

# Train with best C
lasso.C = best_C
lasso.fit(X_train_scaled, y_train)

# Get non-zero coefficients
coefficients = lasso.coef_[0]
non_zero_features = np.abs(coefficients) > 0.001
n_selected = non_zero_features.sum()

print(f'\nFeatures selected: {n_selected} out of 50')
print(f'Features eliminated: {50 - n_selected}')

lasso_f1 = f1_score(y_test, lasso.predict(X_test_scaled))
print(f'\nTest accuracy: {best_score:.4f}')
print(f'Test F1-score: {lasso_f1:.4f}')
print(f'Baseline: {baseline_test_score:.4f}')

# Visualize coefficients
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All coefficients
axes[0].bar(range(50), np.abs(coefficients), color='#9b59b6', alpha=0.7)
axes[0].axhline(y=0.001, color='r', linestyle='--', label='Zero threshold')
axes[0].set_xlabel('Feature Index', fontsize=11)
axes[0].set_ylabel('Absolute Coefficient', fontsize=11)
axes[0].set_title('L1 Regularization Coefficients', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Non-zero coefficients only
selected_indices = np.where(non_zero_features)[0]
selected_coefs = np.abs(coefficients[non_zero_features])
axes[1].barh(range(len(selected_indices)), selected_coefs, color='#e74c3c')
axes[1].set_yticks(range(len(selected_indices)))
axes[1].set_yticklabels([f'Feature_{i}' for i in selected_indices])
axes[1].set_xlabel('Absolute Coefficient', fontsize=11)
axes[1].set_title('Selected Features (Non-zero)', fontsize=12, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Step 8: Compare All Methods

In [ ]:
print('COMPARING ALL FEATURE SELECTION METHODS')
print('='*60)

comparison_df = pd.DataFrame({
    'Method': ['Baseline (All)', 'Feature Importance', 'RFE', 'Correlation', 'L1 Regularization'],
    'Features Used': [50, n_features_95, 15, 15, n_selected],
    'Test Accuracy': [baseline_test_score, baseline_test_score, rfe_test_score, corr_test_score, best_score],
    'F1-Score': [baseline_f1, baseline_f1, rfe_f1, corr_f1, lasso_f1]
})

print('\n' + comparison_df.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
x = np.arange(len(comparison_df))
axes[0].bar(x, comparison_df['Test Accuracy'], color=['#95a5a6', '#3498db', '#2ecc71', '#f39c12', '#9b59b6'])
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison_df['Method'], rotation=45, ha='right')
axes[0].set_ylabel('Test Accuracy', fontsize=11)
axes[0].set_title('Test Accuracy by Method', fontsize=12, fontweight='bold')
axes[0].axhline(y=baseline_test_score, color='r', linestyle='--', alpha=0.5, label='Baseline')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Features used vs accuracy
axes[1].scatter(comparison_df['Features Used'], comparison_df['Test Accuracy'], 
               s=200, c=range(len(comparison_df)), cmap='viridis', edgecolors='black', linewidth=2)
for i, method in enumerate(comparison_df['Method']):
    axes[1].annotate(method, (comparison_df['Features Used'].iloc[i], comparison_df['Test Accuracy'].iloc[i]),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
axes[1].set_xlabel('Number of Features Used', fontsize=11)
axes[1].set_ylabel('Test Accuracy', fontsize=11)
axes[1].set_title('Features vs Performance', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

best_method = comparison_df.loc[comparison_df['Test Accuracy'].idxmax(), 'Method']
print(f'\n🏆 Best method: {best_method}')
print(f'   Accuracy: {comparison_df["Test Accuracy"].max():.4f}')
print(f'   Features used: {comparison_df.loc[comparison_df["Test Accuracy"].idxmax(), "Features Used"]:.0f}')

## Step 9: Performance vs Number of Features

Let's find the optimal number of features by trying different subset sizes.

In [ ]:
print('FINDING OPTIMAL NUMBER OF FEATURES')
print('='*60)

feature_counts = [5, 10, 15, 20, 25, 30, 40, 50]
accuracies = []
f1_scores = []

for k in feature_counts:
    selector = SelectKBest(score_func=f_classif, k=k)
    X_train_k = selector.fit_transform(X_train, y_train)
    X_test_k = selector.transform(X_test)
    
    rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
    rf.fit(X_train_k, y_train)
    
    acc = rf.score(X_test_k, y_test)
    f1 = f1_score(y_test, rf.predict(X_test_k))
    
    accuracies.append(acc)
    f1_scores.append(f1)
    
    print(f'k={k:2d}: Accuracy={acc:.4f}, F1={f1:.4f}')

optimal_k = feature_counts[np.argmax(accuracies)]
print(f'\n✅ Optimal number of features: {optimal_k}')
print(f'   Best accuracy: {max(accuracies):.4f}')
print(f'   Reduction: {50 - optimal_k} features eliminated ({(50-optimal_k)/50*100:.1f}%)')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Performance vs features
axes[0].plot(feature_counts, accuracies, 'o-', linewidth=2, markersize=8, label='Accuracy')
axes[0].plot(feature_counts, f1_scores, 's-', linewidth=2, markersize=8, label='F1-Score')
axes[0].axvline(x=optimal_k, color='r', linestyle='--', label=f'Optimal k={optimal_k}')
axes[0].set_xlabel('Number of Features', fontsize=11)
axes[0].set_ylabel('Score', fontsize=11)
axes[0].set_title('Performance vs Number of Features', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Improvement over baseline
improvements = [(acc - baseline_test_score) * 100 for acc in accuracies]
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in improvements]
axes[1].bar(range(len(feature_counts)), improvements, color=colors, alpha=0.7)
axes[1].set_xticks(range(len(feature_counts)))
axes[1].set_xticklabels(feature_counts)
axes[1].set_xlabel('Number of Features', fontsize=11)
axes[1].set_ylabel('Improvement over Baseline (%)', fontsize=11)
axes[1].set_title('Accuracy Improvement', fontsize=12, fontweight='bold')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: Common Mistakes to Avoid

In [ ]:
print('COMMON FEATURE SELECTION MISTAKES')
print('='*60)

print('\n❌ MISTAKE 1: Selecting features on entire dataset')
print('   Problem: Information from test set leaks into training')
print('   Solution: Fit selector only on training data, transform both')

print('\n❌ MISTAKE 2: Removing features with low variance')
print('   Problem: Low variance ≠ not useful (e.g., rare events)')
print('   Solution: Use domain knowledge and supervised methods')

print('\n❌ MISTAKE 3: Using only correlation with target')
print('   Problem: Misses feature interactions and non-linear relationships')
print('   Solution: Combine multiple selection methods')

print('\n❌ MISTAKE 4: Removing too many features')
print('   Problem: Lose important information, underfitting')
print('   Solution: Use plots to find sweet spot (this notebook!)')

print('\n❌ MISTAKE 5: Not validating after selection')
print('   Problem: Selected features might not generalize')
print('   Solution: Always validate with CV or test set')

print('\n❌ MISTAKE 6: Removing correlated features blindly')
print('   Problem: Correlation ≠ redundancy, both might be useful')
print('   Solution: Remove only if one dominates in importance')

print('\n✅ BEST PRACTICES:')
print('   1. Always fit on training data only')
print('   2. Try multiple selection methods')
print('   3. Plot performance vs number of features')
print('   4. Use CV to validate selection')
print('   5. Consider domain knowledge')
print('   6. Document which features you removed and why')

## Summary: What You Learned

### Key Concepts:

1. **Why Feature Selection**: Reduces overfitting, speeds training, improves interpretability

2. **Four Methods**:
   - Feature Importance (tree-based): Fast, intuitive, model-specific
   - RFE: Recursive elimination, computationally expensive but thorough
   - Correlation-based: Fast, simple, catches linear relationships
   - L1 Regularization: Automatic, built into training, good for linear models

3. **Optimal Features**: Usually 15-30% of total features contain 90-95% of information

4. **Performance**: Feature selection often improves test accuracy by 2-10%

### Method Comparison:

| Method | Speed | Accuracy | When to Use |
|--------|-------|----------|-------------|
| Feature Importance | Fast | Good | Tree-based models |
| RFE | Slow | Best | Small datasets, critical applications |
| Correlation | Very Fast | Good | Quick baseline, linear relationships |
| L1 Regularization | Medium | Good | Linear models, automatic selection |

### Quiz Preparation:

You should now understand:
- How tree-based models calculate feature importance
- How RFE works recursively
- When to use correlation vs mutual information
- How L1 regularization performs feature selection
- How to find the optimal number of features

### Next Steps:

1. Apply to your own datasets
2. Combine multiple methods (ensemble selection)
3. Use domain knowledge to validate selections
4. Try advanced methods (mutual information, SHAP values)
5. Integrate with hyperparameter tuning

> **💡 AI Assistant Challenge:**
> 
> "I have 100 features but only 500 training samples. My model is overfitting. Walk me through feature selection using at least 2 methods. Include code to find the optimal number of features and show before/after performance."

**Great work! You can now select the most important features and build lean, high-performing models!** 🎉